## CPT Processing Guide (TXT + PDF) — OpenStatv2

This notebook documents (and optionally runs) the **existing** OpenStatv2 pipeline for converting scraped **TXT** and downloaded **PDFs** into **CPT (continued pre-training) JSONL**.

Everything here is based on code already in this repo:

- CPT record schema: `src/agri_corpus/cpt_utils.py`
- Chunking: `src/agri_corpus/text_utils.py`
- PDF extraction + noise stripping + cleanup: `src/agri_corpus/pdf_utils.py` (used by `src/services/philrice.py` and `src/services/pinoyrice.py`)
- Source processors:
  - IRRI TXT → CPT: `src/services/irri.py`
  - PhilRice News TXT → CPT: `src/services/philrice_news.py`
  - PinoyRice text + PDFs → CPT: `src/services/pinoyrice.py`
  - PhilRice PDFs → CPT: `src/services/philrice.py`

### What you’ll get

- CPT corpora (JSONL; one JSON object per line)
- Per-file JSON outputs (debuggable per document)

### Important note

- **Cleaning removes noise. Chunking only controls length/shape.**
- CPT is **not** SFT: there is no `instruction`/`output` here—only raw domain text.

## 1) CPT record format (the “standard line”)

OpenStatv2 uses a single CPT JSON object per line.

The record is built by `make_cpt_record()` in `src/agri_corpus/cpt_utils.py`.

### Required fields (always present)

- `text`: the chunk text
- `input`: same as `text`
- `content`: same as `text`
- `source`: source name (e.g. `irri`, `philrice`, `pinoyrice`)
- `doc_id`: stable identifier for the document/chunk

### Optional metadata fields (included by some sources)

- `filename`: original `.txt` or `.pdf` filename
- `page`: PDF page number (for per-page extraction)
- `url`: original page URL (used by PinoyRice text)
- `title`: original page title (used by PinoyRice text)

Why metadata matters:

- It lets you group chunks back to the same document.
- It helps debugging and future RAG (showing citations / provenance).

## 2) Chunking (when enabled)

Chunking is **optional**. By default in this repo, most processors keep one record per document unless you set `*_MAX_CHUNK_CHARS > 0`.

Chunking is implemented in `chunk_text()` in `src/agri_corpus/text_utils.py`:

- If `max_chars <= 0` or the text is already short enough → returns a single chunk.
- Otherwise:
  - Split the text by blank-line paragraphs (`\n\n`).
  - Pack paragraphs into chunks up to `max_chars` (character-based), preserving paragraph boundaries.
  - If one paragraph is longer than `max_chars`, it is split into fixed-length character slices.

### Minimum chunk size filter

Each service also applies a minimum length filter:

- `*_MIN_CHUNK_CHARS` defaults to `100`.
- Any chunk shorter than this is **skipped** (not written to JSONL).

This prevents junk fragments (e.g., a stray caption line) from entering the corpus.

## 3) TXT → CPT (News / articles)

TXT inputs come from scrapers and typically look like:

- A header with `URL: ...` and `Title: ...`
- Followed by the article body

The processor is responsible for:

1) parsing the raw TXT
2) cleaning the body (removing noise lines and site chrome)
3) chunking (optional)
4) writing:
   - corpus JSONL
   - per-file JSON records for inspection

### Implementations in this repo

- IRRI: `src/services/irri.py` (input: `data/irri_processed/text/*.txt` → output: `data/irri_processed/irri_corpus.jsonl`)
- PhilRice News: `src/services/philrice_news.py` (input: `data/philrice_news/*.txt` → output: `philrice_news_processed/philrice_news_corpus.jsonl`)
- PinoyRice text: `src/services/pinoyrice.py` (reads an input JSONL produced by its scraper, then cleans + chunks into CPT)

### Typical noise removed (IRRI example)

IRRI cleaning (inside `src/services/irri.py`) removes items like:

- duplicate title line at top of the body
- standalone date line (e.g. `November 6, 2025`)
- `SHARE`
- footer marker blocks (e.g. starting at `For more information, please contact`)
- any remaining `Related News` section (if present in body text)

The goal is: **the CPT chunk should contain only the article content**.

## 4) PDF → CPT (Reports / knowledge products)

For PDFs, the pipeline is:

1) extract text page-by-page
2) detect repeated noise lines (headers/footers) across pages
3) clean each page’s text using the detected noise patterns
4) emit CPT records per page (default), optionally chunking within a page

### Implementations in this repo

- PhilRice PDFs: `src/services/philrice.py`
- PinoyRice PDFs: `src/services/pinoyrice.py`

Both rely on shared helpers in `src/agri_corpus/pdf_utils.py`:

- `extract_text_from_pdf(...)`
- `detect_noise_lines(pages)`
- `clean_pdf_text(text, noise_lines)`

### What “noise” means here

Noise is typically:

- repeating headers/footers (document title, page numbers, disclaimer lines)
- table-of-contents artifacts
- page decorations that appear on many pages

`detect_noise_lines(...)` looks across many extracted pages and identifies lines that repeat enough to be considered boilerplate, then `clean_pdf_text(...)` removes them during cleaning.

### CPT IDs for PDFs

To preserve page context, the processors use:

- `doc_id = {pdf_stem}_page_{page}`
- plus optional chunk suffix: `{pdf_stem}_page_{page}_chunk_{j}`

Metadata includes:

- `filename` (original PDF)
- `page` (page number)

## 5) Environment variables (how to control chunking + thresholds)

Each processor reads two key knobs:

- `*_MIN_CHUNK_CHARS` (default `100`)
- `*_MAX_CHUNK_CHARS` (default `0` = no chunking)

Examples used in this repo:

- IRRI: `IRRI_MIN_CHUNK_CHARS`, `IRRI_MAX_CHUNK_CHARS`
- PhilRice News: `PHILRICE_NEWS_MIN_CHUNK_CHARS`, `PHILRICE_NEWS_MAX_CHUNK_CHARS`
- PinoyRice: `PINOYRICE_MIN_CHUNK_CHARS`, `PINOYRICE_MAX_CHUNK_CHARS`
- PhilRice PDF: `PHILRICE_MIN_CHUNK_CHARS`, `PHILRICE_MAX_CHUNK_CHARS`

Recommended approach:

- Keep `MIN_CHUNK_CHARS=100` unless you see lots of tiny junk chunks.
- Only set `MAX_CHUNK_CHARS` if you observe very long documents/chunks that you want to split.

For IRRI News specifically, many articles are already short enough for one-chunk CPT records.

## 6) Output locations (current repo conventions)

These are the standard CPT outputs created by the processors:

- PhilRice PDFs:
  - corpus: `philrice_processed/philrice_corpus.jsonl`
  - per-file JSON: `philrice_processed/per_file/*.json`

- PhilRice News:
  - corpus: `philrice_news_processed/philrice_news_corpus.jsonl`
  - per-file JSON: `philrice_news_processed/per_file/*.json`

- PinoyRice:
  - text corpus: `pinoyrice_processed/pinoyrice_corpus.jsonl` (and/or `data/pinoyrice_processed/pinoyrice_corpus.jsonl` depending on run path)
  - PDF corpus: `pinoyrice_processed/pinoyrice_pdfs_corpus.jsonl`
  - per-file JSON:
    - CPT: `pinoyrice_processed/per_file_cpt/*.json`
    - PDF: `pinoyrice_processed/per_file_pdf/*.json`

- IRRI News:
  - corpus: `data/irri_processed/irri_corpus.jsonl`
  - per-file JSON: `data/irri_processed/per_file/*.json`

If you combine corpora for CPT training, you can concatenate JSONL files; each line is an independent CPT record.

In [ ]:
from pathlib import Path
import os
import json

# Repo root = current working directory in most notebook runs
ROOT = Path.cwd()

paths = {
    "irri": ROOT / "data" / "irri_processed" / "irri_corpus.jsonl",
    "philrice": ROOT / "philrice_processed" / "philrice_corpus.jsonl",
    "philrice_news": ROOT / "philrice_news_processed" / "philrice_news_corpus.jsonl",
    "pinoyrice": ROOT / "pinoyrice_processed" / "pinoyrice_corpus.jsonl",
    "pinoyrice_pdfs": ROOT / "pinoyrice_processed" / "pinoyrice_pdfs_corpus.jsonl",
}

def count_jsonl_lines(p: Path) -> int:
    if not p.is_file():
        return 0
    with p.open("r", encoding="utf-8") as f:
        return sum(1 for _ in f)

for name, p in paths.items():
    print(f"{name:14} {count_jsonl_lines(p):6}  ->  {p}")

## 7) Optional: running the processors (uses existing repo code)

If you want to regenerate CPT outputs from your current `data/` directories, run the service modules.

Notes:

- These processors read environment variables from `.env` (via `python-dotenv`).
- They will overwrite their target `*_corpus.jsonl` outputs on each run.

### IRRI

- input: `data/irri_processed/text/*.txt`
- outputs:
  - `data/irri_processed/irri_corpus.jsonl`
  - `data/irri_processed/per_file/*.json`

### PhilRice PDFs

- input: `philrice_pdfs/*.pdf`
- outputs:
  - `philrice_processed/philrice_corpus.jsonl`
  - `philrice_processed/per_file/*.json`

### PhilRice News

- input: `philrice_news/*.txt`
- outputs:
  - `philrice_news_processed/philrice_news_corpus.jsonl`
  - `philrice_news_processed/per_file/*.json`

### PinoyRice

- text corpus input and PDF folders depend on the scraper outputs for PinoyRice.
- outputs are under `pinoyrice_processed/`.

See `docs/DATA_CURRATION.md` for the end-to-end run commands and folder map.

In [ ]:
# Optional execution cell (uncomment the one you want)

# from src.services import irri as processing_irri
# processing_irri.run()

# from src.services import philrice as processing_philrice
# processing_philrice.run()

# from src.services import philrice_news as processing_philrice_news
# processing_philrice_news.run()

# from src.services import pinoyrice as processing_pinoyrice
# processing_pinoyrice.run_text()  # or other entrypoints inside the module


## 8) Practical cleaning checklist (based on current repo behavior)

When reviewing CPT quality, use this checklist per source.

### For TXT/article sources (IRRI / PhilRice News / PinoyRice text)

You want chunks that:

- do **not** contain navigation labels (menu items)
- do **not** contain share widgets (e.g. `SHARE`)
- do **not** contain `Related News` blocks
- do **not** contain contact/footer boilerplate
- do **not** contain duplicated title lines at the start

In this repo:

- IRRI cleaning removes `SHARE`, date line, footer markers, and duplicate title.
- PinoyRice text cleaning aggressively strips noise lines (reply widgets, time-ago stamps, subscribe/footer markers) and unwanted substrings.

### For PDFs

You want page text that:

- does **not** repeat the same header/footer on every page
- does **not** contain page-number artifacts repeated across the whole PDF

In this repo:

- Noise detection is automatic via `detect_noise_lines(...)` across pages.
- Cleaning removes those detected lines via `clean_pdf_text(...)` before writing CPT records.

If you still see boilerplate, it means the line wasn’t frequent enough to be flagged as noise, and you can improve the heuristics in `src/agri_corpus/pdf_utils.py` (without changing the CPT schema).

## 9) Code cells (source-of-truth = repo modules)

The cells below are designed so that future users can:

- run TXT/PDF → CPT using the same repo functions (same outputs)
- view the exact code being executed (pulled directly from the installed repo modules)

This keeps the notebook aligned with the pipeline even if the implementation evolves.

In [ ]:
# Notebook setup: ensure repo root is on sys.path
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Repo root:", ROOT)
print("sys.path[0]:", sys.path[0])

In [ ]:
# Show the exact code used for CPT record creation + chunking
import inspect

from src.agri_corpus.cpt_utils import make_cpt_record
from src.agri_corpus.text_utils import chunk_text

print("=== make_cpt_record (src/agri_corpus/cpt_utils.py) ===")
print(inspect.getsource(make_cpt_record))

print("\n=== chunk_text (src/agri_corpus/text_utils.py) ===")
print(inspect.getsource(chunk_text))

In [ ]:
# Show the exact PDF noise detection + cleaning code used by the pipeline
import inspect

from src.agri_corpus import pdf_utils

print("=== pdf_utils.detect_noise_lines (src/agri_corpus/pdf_utils.py) ===")
print(inspect.getsource(pdf_utils.detect_noise_lines))

print("\n=== pdf_utils.clean_pdf_text (src/agri_corpus/pdf_utils.py) ===")
print(inspect.getsource(pdf_utils.clean_pdf_text))

print("\n=== pdf_utils.extract_text_from_pdf (src/agri_corpus/pdf_utils.py) ===")
print(inspect.getsource(pdf_utils.extract_text_from_pdf))

In [ ]:
# Show the exact IRRI TXT cleaning + TXT->CPT conversion code
import inspect

from src.services import irri as irri_service

# These are the functions that actually parse/clean/build records
print("=== irri_service._clean_irri_text (src/services/irri.py) ===")
print(inspect.getsource(irri_service._clean_irri_text))

print("\n=== irri_service.txt_to_cpt_records (src/services/irri.py) ===")
print(inspect.getsource(irri_service.txt_to_cpt_records))

In [ ]:
# Example: convert ONE IRRI .txt into CPT records (no scraping needed)
from pathlib import Path
import json

sample_txt = ROOT / "data" / "irri_processed" / "text" / "Asia’s agri leaders, experts unite for stronger rice R&D cooperation at 29th CORRA meeting in Uzbekistan _ International.txt"
print("Sample TXT exists:", sample_txt.is_file(), "->", sample_txt)

records = irri_service.txt_to_cpt_records(str(sample_txt))
print("Records produced:", len(records))

# Preview the first record keys and a text snippet
if records:
    r0 = records[0]
    print("Keys:", sorted(r0.keys()))
    print("doc_id:", r0.get("doc_id"))
    print("source:", r0.get("source"))
    print("filename:", r0.get("filename"))
    print("text[:400]:\n", (r0.get("text") or "")[:400])

# Optional: write a one-off JSON file beside the notebook for inspection
out_path = ROOT / "notebooks" / "_sample_irri_records.json"
out_path.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote:", out_path)

In [ ]:
# Example: convert ONE PhilRice PDF into CPT records (no scraping needed)
# This uses the same shared pdf_utils + cleaning logic used by src/services/philrice.py

import os
from src.services import philrice as philrice_service

# Pick any PDF you already have under philrice_pdfs/
pdf_dir = ROOT / "philrice_pdfs"
print("PhilRice PDF dir exists:", pdf_dir.is_dir(), "->", pdf_dir)

pdfs = sorted(pdf_dir.glob("*.pdf"))
print("PDFs found:", len(pdfs))

if not pdfs:
    print("No PDFs found under philrice_pdfs/. Download PDFs first, or point this cell to another PDF path.")
else:
    pdf_path = pdfs[0]
    print("Using PDF:", pdf_path.name)
    recs = philrice_service.process_one_pdf(str(pdf_path))
    print("CPT records produced:", len(recs))
    if recs:
        r0 = recs[0]
        print("Keys:", sorted(r0.keys()))
        print("doc_id:", r0.get("doc_id"))
        print("filename:", r0.get("filename"))
        print("page:", r0.get("page"))
        print("text[:400]:\n", (r0.get("text") or "")[:400])

## 10) Self-contained pipeline code (copy of current OpenStatv2 logic)

This section embeds the pipeline code directly in the notebook so you can share **only this notebook** (without sharing the repo) and still produce the **same CPT JSONL format**.

Notes:

- The code below is copied from the OpenStatv2 modules you are currently using (`src/agri_corpus/*` and `src/services/*`).
- It writes JSONL in the same CPT schema (`text`, `input`, `content`, `source`, `doc_id`, plus optional metadata).
- If you update the repo later, you can refresh this notebook by re-copying the updated functions.

In [ ]:
# --- CPT schema + chunking (copied from src/agri_corpus/*) ---

import re
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse


def make_cpt_record(
    text: str,
    source: str,
    doc_id: str,
    *,
    url: str | None = None,
    title: str | None = None,
    filename: str | None = None,
    page: int | None = None,
) -> dict:
    """Build one CPT chunk record."""
    rec = {
        "text": text,
        "input": text,
        "content": text,
        "source": source,
        "doc_id": doc_id,
    }
    if url is not None:
        rec["url"] = url
    if title is not None:
        rec["title"] = title
    if filename is not None:
        rec["filename"] = filename
    if page is not None:
        rec["page"] = page
    return rec


def chunk_text(text: str, max_chars: int) -> list[str]:
    """
    Split text into chunks of at most max_chars, respecting blank-line paragraphs.
    If max_chars <= 0 or text is shorter, returns [text] (or [] if empty).
    """
    if not text:
        return []
    text = text.strip()
    if not max_chars or max_chars <= 0 or len(text) <= max_chars:
        return [text]

    out: list[str] = []
    paragraphs = text.split("\n\n")
    current = ""
    for p in paragraphs:
        p = p.strip()
        if not p:
            continue
        if len(current) + len(p) + 2 <= max_chars:
            current = (current + "\n\n" + p).strip() if current else p
        else:
            if current:
                out.append(current)
            if len(p) > max_chars:
                for i in range(0, len(p), max_chars):
                    chunk = p[i : i + max_chars].strip()
                    if chunk:
                        out.append(chunk)
                current = ""
            else:
                current = p
    if current:
        out.append(current)
    return out


def safe_id_from_url(url: str, max_len: int = 120, fallback: str = "doc") -> str:
    """Derive a safe doc_id/filename stem from URL path (no scheme, no query)."""
    try:
        parsed = urlparse(url)
        path = (parsed.path or "").strip("/") or fallback
    except Exception:
        path = fallback
    return re.sub(r"[^\w\-.]", "_", path)[:max_len] or fallback


print("Self-contained CPT helpers loaded.")

In [ ]:
# --- PDF utilities (copied from src/agri_corpus/pdf_utils.py) ---

try:
    import fitz  # PyMuPDF
    HAS_PYMUPDF = True
except ImportError:
    HAS_PYMUPDF = False

try:
    import pdfplumber
    HAS_PDFPLUMBER = True
except ImportError:
    HAS_PDFPLUMBER = False

NOISE_LINE_PATTERN = re.compile(r"^\s*\d{1,4}\s*$", re.MULTILINE)


def table_to_text(table: list[list]) -> str:
    """Convert table (list of rows) to readable text (pipe-separated)."""
    if not table:
        return ""
    return "\n".join(" | ".join(str(c or "").strip() for c in row) for row in table)


def extract_tables_from_page(pdf_path: str, page_num: int) -> list[str]:
    """Extract tables from one page using pdfplumber."""
    if not HAS_PDFPLUMBER:
        return []
    out = []
    try:
        with pdfplumber.open(pdf_path) as doc:
            if 1 <= page_num <= len(doc.pages):
                for t in doc.pages[page_num - 1].extract_tables() or []:
                    txt = table_to_text(t)
                    if txt.strip():
                        out.append(txt)
    except Exception:
        pass
    return out


def extract_text_from_pdf(pdf_path: str) -> list[dict]:
    """Extract text per page; append [Table] blocks if pdfplumber available."""
    if not HAS_PYMUPDF:
        return []
    out = []
    try:
        doc = fitz.open(pdf_path)
        for i in range(len(doc)):
            text = doc[i].get_text()
            if HAS_PDFPLUMBER:
                tables = extract_tables_from_page(pdf_path, i + 1)
                if tables:
                    text = (text.strip() if text else "") + "\n\n[Table]\n" + "\n\n[Table]\n".join(tables)
            if text and text.strip():
                out.append({"page": i + 1, "text": text.strip()})
        doc.close()
    except Exception:
        return []
    return out


def detect_noise_lines(
    pages: list[dict],
    min_freq_ratio: float = 0.3,
    max_line_len: int = 120,
) -> set[str]:
    """Detect running headers/footers by frequency across pages."""
    if not pages:
        return set()
    total = len(pages)
    cnt: Counter = Counter()
    for item in pages:
        seen = set()
        for line in item["text"].split("\n"):
            s = line.strip()
            if s and s not in seen:
                cnt[s] += 1
                seen.add(s)
    threshold = max(2, int(total * min_freq_ratio))
    return {line for line, c in cnt.items() if c >= threshold and len(line) <= max_line_len}


def clean_pdf_text(text: str, noise_lines: set[str] | None = None) -> str:
    """Normalize whitespace, remove standalone page numbers and detected noise lines."""
    if not text or not text.strip():
        return ""
    text = re.sub(r"\r\n", "\n", text)
    text = NOISE_LINE_PATTERN.sub("", text)
    if noise_lines:
        for line in noise_lines:
            text = re.sub(r"(?m)^" + re.escape(line) + r"\s*$", "", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


print("Self-contained PDF helpers loaded. HAS_PYMUPDF=", HAS_PYMUPDF, "HAS_PDFPLUMBER=", HAS_PDFPLUMBER)

In [ ]:
# --- IRRI TXT → CPT (copied from src/services/irri.py; simplified to be path-based) ---

IRRI_TITLE_SUFFIX = re.compile(
    r"\s*[_\|]\s*International Rice Research Institute\s*$",
    re.IGNORECASE,
)
FOOTER_START = re.compile(
    r"\n\s*For more information, please contact\s*:?
\s*\n",
    re.IGNORECASE,
)
DATE_LINE = re.compile(
    r"^\s*(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},?\s*\d{4}\s*$",
    re.IGNORECASE,
)


def _parse_txt_content(raw: str) -> tuple[str, str, str]:
    raw = raw.strip()
    url = ""
    title = ""
    body = raw
    if raw.startswith("URL:"):
        lines = raw.split("\n")
        if len(lines) >= 1:
            url = lines[0].replace("URL:", "").strip()
        if len(lines) >= 2 and lines[1].strip().lower().startswith("title:"):
            title = lines[1].replace("Title:", "").strip()
        if len(lines) > 2:
            body = "\n".join(lines[2:]).strip()
    return url, title, body


def _clean_title_for_id(title: str) -> str:
    if not title:
        return "irri_unknown"
    t = IRRI_TITLE_SUFFIX.sub("", title).strip()
    t = re.sub(r"\s+", " ", t)
    return t or "irri_unknown"


def _safe_doc_id(clean_title: str, max_len: int = 100) -> str:
    s = clean_title.lower().strip()
    s = re.sub(r"[^\w\s\-]", " ", s)
    s = re.sub(r"\s+", "_", s).strip("_")
    return (s[:max_len] if max_len else s) or "irri_unknown"


def _clean_body(body: str, clean_title: str = "") -> str:
    if not body:
        return ""
    if FOOTER_START.search(body):
        body = FOOTER_START.split(body)[0].strip()
    lines = body.split("\n")
    out = []
    skip_leading = True
    for ln in lines:
        s = ln.strip()
        if skip_leading:
            if not s:
                continue
            if s.upper() == "SHARE" or DATE_LINE.match(s):
                continue
            if clean_title and s == clean_title:
                continue
            skip_leading = False
        out.append(ln)
    body = "\n".join(out).strip()

    lines = [ln.strip() for ln in body.split("\n")]
    out = []
    prev_blank = False
    for ln in lines:
        is_blank = not ln
        if is_blank and prev_blank:
            continue
        prev_blank = is_blank
        out.append(ln)
    return "\n".join(out).strip()


def clean_irri_txt(raw: str) -> tuple[str, str, str]:
    url, title, body = _parse_txt_content(raw)
    clean_title = _clean_title_for_id(title)
    cleaned_body = _clean_body(body, clean_title=clean_title)
    return url, clean_title, cleaned_body


def irri_txt_to_cpt_records(
    txt_path: str,
    *,
    source_name: str = "irri",
    min_chunk_chars: int = 100,
    max_chunk_chars: int = 0,
) -> list[dict]:
    path = Path(txt_path)
    if not path.is_file() or path.suffix.lower() != ".txt":
        return []
    raw = path.read_text(encoding="utf-8").strip()
    if not raw:
        return []
    url, clean_title, text = clean_irri_txt(raw)
    if not text or len(text) < min_chunk_chars:
        return []
    base_id = _safe_doc_id(clean_title, max_len=100)
    extra = {"filename": path.name}

    if max_chunk_chars > 0 and len(text) > max_chunk_chars:
        chunks = chunk_text(text, max_chunk_chars)
        records = []
        for j, ch in enumerate(chunks):
            if len(ch) < min_chunk_chars:
                continue
            records.append(make_cpt_record(ch, source_name, f"{base_id}_chunk_{j}", **extra))
        return records

    return [make_cpt_record(text, source_name, base_id, **extra)]


print("Self-contained IRRI TXT processor loaded.")

In [ ]:
# --- PhilRice PDF → CPT (copied from src/services/philrice.py; self-contained version) ---


def philrice_pdf_to_cpt_records(
    pdf_path: str,
    *,
    source_name: str = "philrice",
    min_chunk_chars: int = 100,
    max_chunk_chars: int = 0,
    chunk_by_page: bool = True,
) -> list[dict]:
    pages = extract_text_from_pdf(pdf_path)
    if not pages:
        return []
    noise_lines = detect_noise_lines(pages)

    records: list[dict] = []
    base_id = Path(pdf_path).stem
    fname = Path(pdf_path).name

    for item in pages:
        page_num = item["page"]
        text = clean_pdf_text(item["text"], noise_lines)
        if len(text) < min_chunk_chars:
            continue

        if chunk_by_page and not max_chunk_chars:
            records.append(make_cpt_record(text, source_name, f"{base_id}_page_{page_num}", filename=fname, page=page_num))
        else:
            chunks = chunk_text(text, max_chunk_chars) if max_chunk_chars else [text]
            for j, ch in enumerate(chunks):
                if len(ch) < min_chunk_chars:
                    continue
                records.append(make_cpt_record(ch, source_name, f"{base_id}_page_{page_num}_chunk_{j}", filename=fname, page=page_num))

    return records


print("Self-contained PhilRice PDF processor loaded.")

## 11) Run without scraping (folder → CPT JSONL)

The helpers above let you create CPT outputs from existing files on disk.

Typical inputs:

- IRRI TXT folder: `data/irri_processed/text/*.txt`
- PhilRice PDF folder: `philrice_pdfs/*.pdf`

The code cell below writes JSONL outputs similar to the repo processors:

- IRRI: `irri_corpus_from_txt.jsonl`
- PhilRice PDFs: `philrice_corpus_from_pdfs.jsonl`

You can rename these outputs to match whatever your training team expects.

In [ ]:
import json


def write_jsonl(records: list[dict], out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def build_irri_corpus_from_txt_folder(
    txt_dir: Path,
    out_jsonl: Path,
    *,
    min_chunk_chars: int = 100,
    max_chunk_chars: int = 0,
) -> int:
    txt_files = sorted(txt_dir.glob("*.txt"))
    all_records: list[dict] = []
    for p in txt_files:
        all_records.extend(
            irri_txt_to_cpt_records(
                str(p),
                min_chunk_chars=min_chunk_chars,
                max_chunk_chars=max_chunk_chars,
            )
        )
    write_jsonl(all_records, out_jsonl)
    return len(all_records)


def build_philrice_corpus_from_pdf_folder(
    pdf_dir: Path,
    out_jsonl: Path,
    *,
    min_chunk_chars: int = 100,
    max_chunk_chars: int = 0,
    chunk_by_page: bool = True,
) -> int:
    pdf_files = sorted(pdf_dir.glob("*.pdf"))
    all_records: list[dict] = []
    for p in pdf_files:
        all_records.extend(
            philrice_pdf_to_cpt_records(
                str(p),
                min_chunk_chars=min_chunk_chars,
                max_chunk_chars=max_chunk_chars,
                chunk_by_page=chunk_by_page,
            )
        )
    write_jsonl(all_records, out_jsonl)
    return len(all_records)


print("Writers/builders ready.")

In [ ]:
# Example run (edit paths if needed)

irri_txt_dir = Path("data/irri_processed/text")
philrice_pdf_dir = Path("philrice_pdfs")

out_irri = Path("notebooks/output/irri_corpus_from_txt.jsonl")
out_philrice = Path("notebooks/output/philrice_corpus_from_pdfs.jsonl")

print("IRRI TXT dir exists:", irri_txt_dir.is_dir(), irri_txt_dir)
print("PhilRice PDF dir exists:", philrice_pdf_dir.is_dir(), philrice_pdf_dir)

# Adjust these if you want chunking
MIN_CHUNK_CHARS = 100
MAX_CHUNK_CHARS = 0  # set >0 to enable chunking

if irri_txt_dir.is_dir():
    n = build_irri_corpus_from_txt_folder(irri_txt_dir, out_irri, min_chunk_chars=MIN_CHUNK_CHARS, max_chunk_chars=MAX_CHUNK_CHARS)
    print("Wrote", n, "records ->", out_irri)

if philrice_pdf_dir.is_dir():
    if not HAS_PYMUPDF:
        print("PyMuPDF not installed; PDF extraction will not work. Install: pip install pymupdf")
    else:
        n = build_philrice_corpus_from_pdf_folder(philrice_pdf_dir, out_philrice, min_chunk_chars=MIN_CHUNK_CHARS, max_chunk_chars=MAX_CHUNK_CHARS, chunk_by_page=True)
        print("Wrote", n, "records ->", out_philrice)